### Installing dependices

In [11]:
%pip install numpy pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


### Part A — Designing the dataset
 # Topic :Fitness: Predict injury risk from training volume, sleep hours, protein intake


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300

# Feature Generation
training_volume = np.random.uniform(2, 20, n)      # hours/week
sleep_hours     = np.random.uniform(4, 10, n)      # hours/night
protein_intake  = np.random.uniform(40, 200, n)    # grams/day

# Label Logic
score = (
    0.5 * (training_volume - 2) / 18
  - 0.3 * (sleep_hours - 4) / 6
  - 0.2 * (protein_intake - 40) / 160
  + np.random.normal(0, 0.13, n)
)

label = np.where(score > 0.2, 'at_risk', 'safe')

df = pd.DataFrame({
    'training_volume_hrs' : np.round(training_volume, 1),
    'sleep_hours'         : np.round(sleep_hours, 1),
    'protein_intake_g'    : np.round(protein_intake, 0).astype(int),
    'injury_risk'         : label
})

print("Class distribution:")
print(df['injury_risk'].value_counts())
print(f"\nGenerated {len(df)} records")

df.to_csv('my_dataset.csv', index=False)
print("Saved to my_dataset.csv")


Class distribution:
injury_risk
safe       237
at_risk     63
Name: count, dtype: int64

Generated 300 records
Saved → my_dataset.csv


### part B:Train & Evaluate Pipeline


In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# ── Load Data ────────────────────────────────────────────────────────────────
df = pd.read_csv("my_dataset.csv")
print("Dataset shape:", df.shape)
print(df.head())

X = df[["training_volume_hrs", "sleep_hours", "protein_intake_g"]].values
y = df["injury_risk"].values

# ── Encode Labels ────────────────────────────────────────────────────────────
le = LabelEncoder()
y_enc = le.fit_transform(y)  # at_risk=0, safe=1
print("\nLabel encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── 80/20 Train-Test Split ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"\nTrain size: {len(X_train)} | Test size: {len(X_test)}")

# ── Feature Scaling (fit on TRAIN only) ──────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ── Choose K = √N rounded to nearest odd ─────────────────────────────────────
import math

k_raw = math.sqrt(len(X_train))
k = int(round(k_raw))
if k % 2 == 0:
    k += 1
print(f"\nK value: {k} (sqrt of {len(X_train)})")

# ── Train KNN ────────────────────────────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train_scaled, y_train)

# ── Evaluate ─────────────────────────────────────────────────────────────────
y_pred = knn.predict(X_test_scaled)

print("\nCLASSIFICATION REPORT")
print("-" * 45)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")

# ── Predict on 3 Manual Points ───────────────────────────────────────────────
manual_points = np.array(
    [
        [18.0, 5.0, 50],
        [5.0, 9.0, 180],
        [12.0, 7.0, 110],
    ]
)
manual_scaled = scaler.transform(manual_points)
manual_preds = le.inverse_transform(knn.predict(manual_scaled))
manual_proba = knn.predict_proba(manual_scaled)


Dataset shape: (300, 4)
   training_volume_hrs  sleep_hours  protein_intake_g injury_risk
0                  8.7          4.3                67        safe
1                 19.1          7.2                85        safe
2                 15.2          7.2                68     at_risk
3                 12.8          7.8                54        safe
4                  4.8          8.4                59        safe

Label encoding: {'at_risk': np.int64(0), 'safe': np.int64(1)}

Train size: 240 | Test size: 60

K value: 15 (sqrt of 240)

CLASSIFICATION REPORT
---------------------------------------------
              precision    recall  f1-score   support

     at_risk       0.56      0.77      0.65        13
        safe       0.93      0.83      0.88        47

    accuracy                           0.82        60
   macro avg       0.74      0.80      0.76        60
weighted avg       0.85      0.82      0.83        60

Accuracy Score: 0.8167


In [20]:
print("\nMANUAL PREDICTION ON 3 CUSTOM POINTS")
print("-" * 50)
labels_ordered = le.classes_
for i, (point, pred, proba) in enumerate(
    zip(manual_points, manual_preds, manual_proba)
):
    print(
        f"\nPoint {i+1}: training={point[0]}h/wk | sleep={point[1]}h | protein={point[2]}g"
    )
    print(f"Prediction: {pred}")
    for cls, p in zip(labels_ordered, proba):
        print(f"  {cls}: {p:.2f}")



MANUAL PREDICTION ON 3 CUSTOM POINTS
--------------------------------------------------

Point 1: training=18.0h/wk | sleep=5.0h | protein=50.0g
Prediction: at_risk
  at_risk: 0.73
  safe: 0.27

Point 2: training=5.0h/wk | sleep=9.0h | protein=180.0g
Prediction: safe
  at_risk: 0.00
  safe: 1.00

Point 3: training=12.0h/wk | sleep=7.0h | protein=110.0g
Prediction: safe
  at_risk: 0.33
  safe: 0.67


Saving the scaler model of knn 

In [ ]:
import pickle, os
os.makedirs("model", exist_ok=True)
with open("model/knn_model.pkl", "wb") as f:
    pickle.dump(
        {
            "knn": knn,
            "scaler": scaler,
            "le": le,
            "k": k,
            "X_train_scaled": X_train_scaled,
            "X_test_scaled": X_test_scaled,
            "y_train": y_train,
            "y_test": y_test,
        },
        f,
    )
print("Model saved to model/knn_model.pkl")



Model saved → model/knn_model.pkl


### Part C: Visualization Portfolio


In [22]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# ── Load saved model and data ─────────────────────────────────────────────────
with open('model/knn_model.pkl', 'rb') as f:
    saved = pickle.load(f)

knn            = saved['knn']
scaler         = saved['scaler']
le             = saved['le']
k              = saved['k']
X_train_scaled = saved['X_train_scaled']
X_test_scaled  = saved['X_test_scaled']
y_train        = saved['y_train']
y_test         = saved['y_test']

df = pd.read_csv('my_dataset.csv')
X  = df[['training_volume_hrs', 'sleep_hours', 'protein_intake_g']].values
y  = le.transform(df['injury_risk'].values)

# ── Shared style ──────────────────────────────────────────────────────────────
BG       = '#0e0e14'
SURFACE  = '#16161f'
C0       = '#e05c6c'   # at_risk  → red
C1       = '#22d3a7'   # safe     → teal
ACCENT   = '#7c5cfc'
TEXT     = '#d8d6e8'
GRID     = '#2a2a3d'

plt.rcParams.update({
    'figure.facecolor' : BG,
    'axes.facecolor'   : SURFACE,
    'axes.edgecolor'   : GRID,
    'axes.labelcolor'  : TEXT,
    'xtick.color'      : TEXT,
    'ytick.color'      : TEXT,
    'text.color'       : TEXT,
    'grid.color'       : GRID,
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'font.family'      : 'monospace',
})









### VIZ 1 — Decision Boundary Map (features 0 & 1: training_volume vs sleep)




In [16]:
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor(BG)

feat_idx = [0, 1]
feat_names = ["Training Volume (scaled)", "Sleep Hours (scaled)"]

X2d_train = X_train_scaled[:, feat_idx]
X2d_test = X_test_scaled[:, feat_idx]

knn_2d = KNeighborsClassifier(n_neighbors=k)
knn_2d.fit(X2d_train, y_train)

h = 0.04
x_min = X2d_train[:, 0].min() - 1.2
x_max = X2d_train[:, 0].max() + 1.2
y_min = X2d_train[:, 1].min() - 1.2
y_max = X2d_train[:, 1].max() + 1.2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
cmap_bg = ListedColormap([C0 + "28", C1 + "28"])
ax.contourf(xx, yy, Z, cmap=cmap_bg, alpha=1.0)
ax.contour(xx, yy, Z, colors=[GRID], linewidths=1.2, alpha=0.6)

for cls, color, label in zip([0, 1], [C0, C1], le.classes_):
    mask = y_train == cls
    ax.scatter(
        X2d_train[mask, 0],
        X2d_train[mask, 1],
        c=color,
        edgecolors="white",
        linewidths=0.4,
        s=45,
        alpha=0.85,
        label=f"Train: {label}",
        zorder=3,
    )
    mask_t = y_test == cls
    ax.scatter(
        X2d_test[mask_t, 0],
        X2d_test[mask_t, 1],
        c=color,
        edgecolors="white",
        linewidths=0.4,
        s=45,
        alpha=0.85,
        marker="D",
        zorder=3,
    )

ax.set_xlabel(feat_names[0], fontsize=11)
ax.set_ylabel(feat_names[1], fontsize=11)
ax.set_title(
    f"KNN Decision Boundary  (K={k})\nFitness Injury Risk Classifier",
    fontsize=13,
    fontweight="bold",
    pad=14,
)
ax.legend(loc="upper right", framealpha=0.2, fontsize=9)
ax.grid(True)

plt.tight_layout()
plt.savefig("viz1_decision_boundary.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("Saved: viz1_decision_boundary.png")


✅ viz1_decision_boundary.png saved


### VIZ 2 — K vs Accuracy Curve

In [17]:

k_range = range(1, 50, 2)
train_acc = []
test_acc = []

for ki in k_range:
    m = KNeighborsClassifier(n_neighbors=ki)
    m.fit(X_train_scaled, y_train)
    train_acc.append(accuracy_score(y_train, m.predict(X_train_scaled)))
    test_acc.append(accuracy_score(y_test, m.predict(X_test_scaled)))

best_k = list(k_range)[np.argmax(test_acc)]
best_acc = max(test_acc)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)

ax.plot(
    list(k_range),
    train_acc,
    "o-",
    color=ACCENT,
    lw=2,
    ms=5,
    label="Train Accuracy",
    alpha=0.9,
)
ax.plot(
    list(k_range),
    test_acc,
    "s-",
    color=C1,
    lw=2,
    ms=5,
    label="Test Accuracy",
    alpha=0.9,
)
ax.axvline(
    best_k,
    color="#f5a623",
    linestyle="--",
    lw=1.5,
    label=f"Optimal K={best_k}  (acc={best_acc:.3f})",
)
ax.axvline(
    k, color="white", linestyle=":", lw=1.5, label=f"Used K={k}  (sqrt N rule)", alpha=0.6
)

ax.fill_between(list(k_range), train_acc, test_acc, alpha=0.06, color=ACCENT)
ax.set_xlabel("K Value", fontsize=11)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_title(
    "K vs Accuracy - Bias-Variance Tradeoff",
    fontsize=13,
    fontweight="bold",
)
ax.legend(framealpha=0.2, fontsize=9)
ax.set_ylim(0.4, 1.05)
ax.grid(True)

plt.tight_layout()
plt.savefig("viz2_k_accuracy_curve.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("Saved: viz2_k_accuracy_curve.png")


✅ viz2_k_accuracy_curve.png saved


3.  VIZ 3 — Feature Distribution Matrix (manual pairplot)


In [ ]:
features = ["training_volume_hrs", "sleep_hours", "protein_intake_g"]
feat_labels = ["Training Vol (hrs)", "Sleep (hrs)", "Protein (g)"]
class_labels = le.classes_
colors_map = {cls: c for cls, c in zip(class_labels, [C0, C1])}

fig, axes = plt.subplots(3, 3, figsize=(11, 10))
fig.patch.set_facecolor(BG)
fig.suptitle(
    "Feature Distribution Matrix",
    fontsize=13,
    fontweight="bold",
    y=1.01,
)

for i, (fi, li) in enumerate(zip(features, feat_labels)):
    for j, (fj, lj) in enumerate(zip(features, feat_labels)):
        ax = axes[i][j]
        ax.set_facecolor(SURFACE)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID)
        ax.tick_params(colors=TEXT, labelsize=7)

        if i == j:
            # Diagonal: histogram per class
            for cls in class_labels:
                vals = df.loc[df["injury_risk"] == cls, fi]
                ax.hist(
                    vals,
                    bins=18,
                    color=colors_map[cls],
                    alpha=0.55,
                    edgecolor="none",
                    density=True,
                )
            ax.set_ylabel(li, fontsize=8, color=TEXT)
        else:
            for cls in class_labels:
                mask = df["injury_risk"] == cls
                ax.scatter(
                    df.loc[mask, fj],
                    df.loc[mask, fi],
                    c=colors_map[cls],
                    s=10,
                    alpha=0.5,
                    edgecolors="none",
                )
            if i == 2:
                ax.set_xlabel(lj, fontsize=8, color=TEXT)
            if j == 0:
                ax.set_ylabel(li, fontsize=8, color=TEXT)

patch0 = mpatches.Patch(color=C0, label="at_risk")
patch1 = mpatches.Patch(color=C1, label="safe")
fig.legend(
    handles=[patch0, patch1], loc="lower right", framealpha=0.2, fontsize=9, ncol=2
)

plt.tight_layout()
plt.savefig("viz3_feature_distribution.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("Saved: viz3_feature_distribution.png")


✅ viz3_feature_distribution.png saved
